In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [6]:
from rag_helper import RAGBase

instructions = """

You are a course teaching assistant.
Answer the question based on CONTEXT from the FAQ Database.
Use only the facts from the CONTEXT when answering the question.
""".strip()

assistant = RAGBase(
    index= index,
    llm_client=openai_client,
    instructions=instructions,
)

In [7]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your OS.
2. Open a terminal and run:

```bash
ollama run llama3
```

This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

To test that the local server is running, you can also run:

```bash
curl http://localhost:11434
```

If you get a connection refused error while using Ollama in the homework, restart the server with:

```bash
!nohup ollama serve > nohup.out 2>&1 &
```


In [8]:
messages = [
    {'role' : 'user', 'content': 'I just discovered the course, can I join it?'
    }
]

response = openai_client.responses.create(
    model = 'gpt-5.4-mini',
    input = messages,
)

response.output_text

'Likely yes — but it depends on the course’s enrollment policy and whether there are still seats open.\n\nIf you want, I can help you check or draft a message to the instructor/organizer. A simple version would be:\n\n> Hi, I just discovered the course and I’m very interested in joining. Is it still possible to enroll? If so, please let me know the next steps. Thank you!\n\nIf you tell me the course name and where it’s hosted, I can help you make the message more specific.'

In [9]:
def search(query):
    boost_dict = {'question' : 3.0, 'section' : 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict= boost_dict,
        filter_dict= filter_dict
    )

In [10]:
index.search("How to run ollama?")

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [11]:
search_tool = {
    'type' : 'function',
    'name' : 'search',
    'description' : 'Search the FAQ database for entries matching the given query.',
    'parameters' : {
        'type' :'object',
        'properties' :{
            'query' : {
                'type' : 'string',
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        'required':['query'],
        'additionalProperties' : False
    }
}

In [12]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input = messages,
    tools = [search_tool]

)

In [13]:
len(response.output)

1

In [14]:
call = response.output[0]

In [15]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join course"}', call_id='call_YTWpCBXp1rGRAQ9h9tgMzmRL', name='search', type='function_call', id='fc_0449534cd2abe2b3006a3e5a64b198819eb8592731fb020d2c', namespace=None, status='completed')

In [16]:
import json

args = json.loads(call.arguments)

In [17]:
results = search(**args)

In [18]:
result_json = json.dumps(results, indent=2)

In [19]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."
  },
  {
    "id": "bd31146b0e",
    "course": "llm-zoomcamp",
    "section": "General Course

In [20]:
function_call_output = {
    'type':'function_call_output',
    'call_id' : call.call_id,
    'output' : result_json
}

In [21]:
messages.append(call)

In [22]:
messages.append(function_call_output)

In [23]:
messages

[{'role': 'user', 'content': 'I just discovered the course, can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join course"}', call_id='call_YTWpCBXp1rGRAQ9h9tgMzmRL', name='search', type='function_call', id='fc_0449534cd2abe2b3006a3e5a64b198819eb8592731fb020d2c', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_YTWpCBXp1rGRAQ9h9tgMzmRL',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\

In [24]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [25]:
print(response.output_text)

Yes — you can still join and start learning.

If you want a certificate, though, you need to submit your project while the course is still accepting submissions.


In [26]:
usage = response.usage

usage.input_tokens, usage.output_tokens

(770, 36)

In [27]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [28]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [29]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment late registration FAQ"}
function_call: search {"query":"new student can join course after start enrollment FAQ"}


In [30]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late registration FAQ"}', call_id='call_l2lRRi94ThzSJfsA52Nrk20X', name='search', type='function_call', id='fc_039e5da462b853fc006a3e5a684528819fb76bcc72fd1a946a', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"new student can join course after start enrollment FAQ"}', call_id='ca

In [31]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. If you’re just following along, you can start learning at any time.

If you want, I can also help with the next steps for getting started. Are there other areas you want to explore?


In [32]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [33]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run Ollama locally install run local model"}
iteration #2...
function_call: search {"query":"Ollama serve localhost 11434 python ollama chat model llama3 local server"}
iteration #3...
ASSISTANT:
To run **Ollama locally**, do this:

1. **Install Ollama**
   - macOS: download from https://ollama.com/download and install the `.pkg`
   - Windows: download the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This downloads the model and opens a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response showing the server is up.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "

'To run **Ollama locally**, do this:\n\n1. **Install Ollama**\n   - macOS: download from https://ollama.com/download and install the `.pkg`\n   - Windows: download the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and opens a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response showing the server is up.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error, restart the server with:\n```bash\nollama serve\n```\nor in a notebook:\n```bash\n!nohup ollama serve > 

In [34]:
#encouraging multiple searchees
 
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course enroll late discovered the course can I join FAQ"}
iteration #2...
function_call: search {"query":"certificate project while accepting submissions join course late certificate live cohort self-paced"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If your goal is just to learn, you can start anytime. If you want a certificate, you’ll need to submit your project while submissions are still being accepted.

If you want, I can also help you figure out the best way to start the course now.


'Yes — you can still join the course.\n\nIf your goal is just to learn, you can start anytime. If you want a certificate, you’ll need to submit your project while submissions are still being accepted.\n\nIf you want, I can also help you figure out the best way to start the course now.'

In [35]:
#restricting off topic questions
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
function_call: search {"query":"queen gambit queen's gambit off topic chess opening"}
iteration #4...
ASSISTANT:
I couldn’t find a course FAQ entry for “queen’s gambit,” so it looks like this isn’t covered by the course materials and may be off-topic here.

If you meant something from the course, feel free to rephrase with a course-related term or topic. Is there another area you want to explore?


'I couldn’t find a course FAQ entry for “queen’s gambit,” so it looks like this isn’t covered by the course materials and may be off-topic here.\n\nIf you meant something from the course, feel free to rephrase with a course-related term or topic. Is there another area you want to explore?'

In [36]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [37]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [38]:
search_tool

{'type': 'function',
 'name': 'search',
 'description': 'Search the FAQ database for entries matching the given query.',
 'parameters': {'type': 'object',
  'properties': {'query': {'type': 'string',
    'description': 'Search query text to look up in the course FAQ.'}},
  'required': ['query'],
  'additionalProperties': False}}

In [40]:
def search(query : str) -> dict[str, str]:

    """Search the FAQ databse for entries matching the given query."""

    return index.search(
        query,
        num_results =5,
        boost_dict = {'question' : 3.0, 'section ': 0.5 },
        filter_dict= {'course' : 'llm-zoomcamp'}
    )

In [42]:
agaent_tools = Tools()
agent_tools.add_tool(search)

In [41]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [45]:
chat_interface= IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [44]:
runner = OpenAIResponsesRunner(
    tools = agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model ='gpt-5.4-mini')
)

In [46]:
result = runner.loop(
    prompt='How do I run Ollama locally?',
    callback=callback,
)

-> Response received


-> Response received


In [47]:
result.cost

CostInfo(input_cost=Decimal('0.0010785'), output_cost=Decimal('0.001314'), total_cost=Decimal('0.0023925'))

In [48]:
result2 = runner.loop(
    prompt= 'How do I run a different model?',
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [49]:
runner.run();

-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='how do I run docker in docker', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"docker in docker run docker in docker 